# 02 · Preprocesamiento

**SIRAP** · Proyecto Integrador en IA (MIAR0545) · Eduardo Alejandro Ceballos Jijón · Christian Xavier Valle Maridueña

Prepara el dataset para Ultralytics (copia local, `data.yaml` con rutas absolutas, muestras fijas del preset `rapido`) y define la normalización de placas ecuatorianas (`XXX-0000`, correcciones por posición) con sus pruebas.

> Este notebook es un extracto de `SIRAP_backend.ipynb` (el notebook completo, que es la fuente de verdad). Las celdas y sus salidas son las mismas; la sección 0 se repite para que corra por sí solo en Colab.

## 0. Preparación

Esta sección deja lista la sesión de Colab: comprueba la GPU, instala las librerías, lee las llaves de API, monta Google Drive, define todas las rutas de entrada y fija la configuración del experimento. Hay que correrla completa al empezar cada sesión; no entrena ni evalúa nada.

### GPU
`nvidia-smi` muestra la GPU asignada. Si falla, ve a `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `GPU T4` y guarda. Sin GPU el notebook funciona, pero entrenar y evaluar los detectores se vuelve muy lento (el OCR corre en la CPU en cualquier caso).

In [ ]:
!nvidia-smi

Sat Oct 24 00:16:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   33C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Dependencias
Instala solo las librerías que Colab no trae, en las versiones con las que se probó el notebook, para que los resultados se puedan reproducir:

- `ultralytics==8.4.177`: entrenamiento, validación e inferencia de YOLO11.
- `trackers==2.6.1`: OC-SORT, el seguidor de placas del pipeline de video.
- `supervision==0.30.9`: detecciones, anotación de video y mAP por tamaño de placa.
- `fast-plate-ocr==1.1.0` y `onnxruntime==1.31.0`: el OCR de placas (`cct-s-v2-global-model`) y el motor que lo corre en la CPU. fast-plate-ocr se instala sin sus dependencias (`--no-deps`): pide `opencv-python-headless`, que podría chocar con el `opencv-python` de Colab (dos paquetes para el mismo `cv2`), y las demás (`numpy`, `pyyaml`, `rich` y `tqdm`) ya vienen en Colab.
- `roboflow==1.7.1`: descarga del dataset.
- `Levenshtein`: distancia entre placas (regla de decisión y CER).
- `psycopg[binary]==3.3.6`: conexión con PostgreSQL, donde se carga la base de datos de la planta.

`torch`, `torchvision`, `opencv`, `pandas` y `matplotlib` ya vienen en Colab y no se tocan. Si al terminar Colab pide **reiniciar la sesión**, reinicia y sigue desde la celda siguiente (no hace falta reinstalar).

La celda de importaciones, más abajo, carga todas las librerías y muestra la GPU y las versiones instaladas.

In [ ]:
!pip install -q "ultralytics==8.4.177" "trackers==2.6.1" "supervision==0.30.9" "roboflow==1.7.1" Levenshtein "psycopg[binary]==3.3.6" "onnxruntime==1.31.0"
!pip install -q --no-deps "fast-plate-ocr==1.1.0"  # sin dependencias: usa el OpenCV que ya trae Colab

### Llaves de API (*Secrets* de Colab)

Las llaves no se escriben en el notebook: la función `get_secret` las lee del panel 🔑 **Secrets** de Colab (fuera de Colab, de variables de entorno).

| Secret | Para qué | ¿Obligatorio? |
|---|---|---|
| `ROBOFLOW_API_KEY` | Descargar el dataset de placas desde Roboflow (Roboflow → `Settings` → `API`) | Sí, con `DATASET_SOURCE = "roboflow"` |
| `PG_DSN` | Usar un servidor PostgreSQL propio (por ejemplo Supabase o Neon) en lugar del que el notebook instala en Colab | No (sección 6) |
| `CAMARA_URL` | Dirección de la cámara para el modo en vivo: la del túnel de Cloudflare (`https://….trycloudflare.com`) | No (sección 9) |
| `CAMARA_TOKEN` | Token que imprime `puente_camara.py` en la PC de la cámara | Con el puente (sección 9) |
| `CF_ACCESS_CLIENT_ID` y `CF_ACCESS_CLIENT_SECRET` | Token de servicio, si el túnel está protegido con Cloudflare Access | No (sección 9) |

En Colab: panel izquierdo → 🔑 **Secrets** → agrega cada nombre con su valor y activa el acceso desde el notebook.

> ⚠️ La v3.5 tenía dos llaves de Roboflow escritas en celdas, así que quedaron guardadas en el archivo y en cualquier copia que se haya compartido. **Revócalas en Roboflow (`Settings` → `API`), genera una nueva** y usa solo el *Secret*.

In [ ]:
import os


def get_secret(name: str, required: bool = False):
    """Lee un secreto de Colab (panel 🔑) o, fuera de Colab, de una variable de entorno."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except ImportError:
        value = os.environ.get(name)
    except Exception:  # el secreto no existe o el notebook no tiene acceso
        value = None
    if required and not value:
        raise RuntimeError(f"Falta el secreto {name}: agrégalo en el panel 🔑 Secrets de Colab.")
    return value

In [ ]:
import csv
import copy
import gc
import hashlib
import json
import random
import re
import shutil
import subprocess
import time
import urllib.request
import zipfile
from importlib.metadata import version
from math import comb
from pathlib import Path

import cv2
import Levenshtein
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import supervision as sv
import torch
import yaml
from IPython.display import Image as IPyImage, Video, display
from PIL import Image, ImageDraw, ImageFont
from supervision.metrics import MeanAveragePrecision
from tqdm.auto import tqdm
from trackers import OCSORTTracker
from ultralytics import YOLO
from ultralytics.data.utils import IMG_FORMATS
from ultralytics.utils.torch_utils import get_flops, get_num_params

CUDA = torch.cuda.is_available()
DEVICE = 0 if CUDA else "cpu"
FP16 = 16 if CUDA else None  # inferencia en FP16 en la GPU (argumento `quantize` de Ultralytics 8.4)

print("GPU:", torch.cuda.get_device_name(0) if CUDA else "no disponible: todo será muy lento")
for package in ["torch", "ultralytics", "trackers", "supervision", "fast-plate-ocr", "onnxruntime"]:
    print(f"{package:<16}{version(package)}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
GPU: NVIDIA L4
torch           2.11.0+cu130
ultralytics     8.4.177
trackers        2.6.1
supervision     0.30.9
fast-plate-ocr  1.1.0
onnxruntime     1.31.0


### Carpeta de trabajo en Google Drive
Colab borra `/content` al cerrar la sesión y entrenar el detector toma tiempo, así que todo se guarda en `MyDrive/proyecto_integrador_placas`. Si la sesión se cae a mitad de un entrenamiento, al volver a correr la celda de ese modelo **se reanuda desde la última época guardada**; y los resultados finales (sección 8) se pueden correr en otra sesión, porque lee todo desde aquí.

La celda siguiente monta Drive, define **todas** las rutas de entrada en un solo lugar y al final muestra cuáles encontró (`ok`, `vacío` o `no está`). Las que faltan no detienen el notebook: la sección que usa cada una se salta.

| Ruta en `MyDrive/proyecto_integrador_placas/` | Para qué | Sección |
|---|---|---|
| `savh_ground_truth_subset.zip` | 739 recortes de SAVH Guayaquil con la lectura de la cámara en `ground_truth.csv` | 5 |
| Hoja de Google **Revisión placas SAVH (142 legibles)** | Correcciones de esa referencia (se lee con tu cuenta de Google) | 5 |
| `revision_placas_142.csv` | Copia en CSV de la hoja, solo si no se puede leer la hoja (Archivo → Descargar → CSV) | 5 |
| `ocr/savh_ocr_salidas.csv` | Lecturas de GLM-OCR en SAVH que guardaron la v3.14 y anteriores: la sección 5 las compara con fast-plate-ocr | 5 |
| `authorized_plates.csv` | Placas autorizadas fuera de la base de la planta (excepciones), una por línea | 5 y 6 |
| `camiones_postgres.sql` | Base de la planta: vehículos, estado y transportista. Se carga en PostgreSQL | 6 |
| `video_camiones.mp4` | Video real del punto de control, con `USE_SAMPLE_VIDEO = False` | 6 |
| `video_ground_truth.csv` | Referencia anotada del video (placa e intervalo de cuadros) | 7a |
| `garita/imagenes/` y `garita/ground_truth.csv` | Imágenes completas de la garita con la placa transcrita | 7b |
| `dataset_placas_yolo.zip` | Dataset propio en formato YOLO, si `DATASET_SOURCE = "local"` | 1 |

`SAVH_Guayaquil_ANPR_Plate_Images_Dataset.zip` y `SAVH_YOLOv8CLS_Plate_Legibility_Dataset.zip.zip` también están en la carpeta, pero este notebook no los usa: el primero tiene las 2000 imágenes de las que sale el subconjunto, y el segundo es el dataset de legibilidad derivado de LPLCv2 (placas brasileñas). Tampoco usa `camiones.duckdb`, la misma base en DuckDB de versiones anteriores: la base de la planta se trabaja solo en PostgreSQL.

**Salidas.** Cada experimento escribe en `experimentos/<dataset>/`: `runs/` (entrenamientos y validaciones de Ultralytics), `resultados/` (métricas en JSON y CSV, y registros de decisiones), `figuras/` (gráficos para el informe) y `videos/` (videos anotados). La evaluación del OCR va a `ocr/`; la base de la planta se actualiza en su propio archivo, con una copia del original en `respaldo_bd/`. El modo en vivo (sección 9) guarda en `en_vivo/` el script del puente para la PC de la cámara y, por cada sesión, la grabación, el registro de decisiones y el resumen.

**Para retomar en una sesión nueva:** corre la sección 0 completa (no reentrena: los pesos ya están en Drive) y después las celdas que pide la sección que necesites, según la tabla de la portada (**Cómo usarlo**).

In [ ]:
USE_DRIVE = True  # guarda todo en Google Drive

WORK_DIR = Path("/content/proyecto_integrador_placas")
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_DIR = Path("/content/drive/MyDrive/proyecto_integrador_placas")
    except ImportError:
        print("Fuera de Colab: se usa una carpeta local.")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", WORK_DIR)

# ===================== Archivos de entrada (todas las rutas en un solo lugar) =====================
SAVH_ZIP_CANDIDATES = [WORK_DIR / "savh_ground_truth_subset.zip", Path("/content/savh_ground_truth_subset.zip")]
REVIEW_SHEET_ID = "1l79GrAYGFkTiOYxgKDT1_BEi_b-6Duo1_QqUNlcv7yM"  # hoja «Revisión placas SAVH (142 legibles)»
REVIEW_CSV_FALLBACK = WORK_DIR / "revision_placas_142.csv"        # copia en CSV de la hoja (opcional)
GLM_OCR_CACHE_PATH = WORK_DIR / "ocr" / "savh_ocr_salidas.csv"    # lecturas de GLM-OCR de la v3.14 (comparación)
AUTHORIZED_LIST_PATH = WORK_DIR / "authorized_plates.csv"
PLANT_POSTGRES_SQL_PATH = WORK_DIR / "camiones_postgres.sql"      # base de la planta (sección 6), se carga en PostgreSQL
REAL_VIDEO_PATH = WORK_DIR / "video_camiones.mp4"                 # video real del punto de control
VIDEO_GROUND_TRUTH_PATH = WORK_DIR / "video_ground_truth.csv"     # columnas: plate_gt, frame_start, frame_end
E2E_IMAGES_DIR = WORK_DIR / "garita" / "imagenes"                 # imágenes completas de la garita
E2E_GROUND_TRUTH_CSV = WORK_DIR / "garita" / "ground_truth.csv"   # columnas: image_path, plate_gt
LOCAL_DATASET_ZIP = WORK_DIR / "dataset_placas_yolo.zip"          # solo con DATASET_SOURCE = "local"
LIVE_DIR = WORK_DIR / "en_vivo"                                   # modo en vivo (sección 9): puente, grabaciones y registros
BRIDGE_SCRIPT_PATH = LIVE_DIR / "puente_camara.py"                # script para la PC de la cámara (celda 9.1)
LIVE_DIR.mkdir(exist_ok=True)


def input_status(path: Path) -> str:
    if not path.exists():
        return "no está"
    if path.is_file() and path.stat().st_size == 0:
        return "vacío"
    return "ok"


savh_zip_path = next((p for p in SAVH_ZIP_CANDIDATES if p.exists()), SAVH_ZIP_CANDIDATES[0])
display(pd.DataFrame(
    [{"entrada": label, "estado": input_status(path), "ruta": str(path)} for label, path in (
        ("SAVH: recortes y ground_truth.csv", savh_zip_path),
        ("Revisión de placas: copia CSV (opcional)", REVIEW_CSV_FALLBACK),
        ("SAVH: lecturas de GLM-OCR (comparación, opcional)", GLM_OCR_CACHE_PATH),
        ("Placas autorizadas (excepciones)", AUTHORIZED_LIST_PATH),
        ("Base de la planta: PostgreSQL", PLANT_POSTGRES_SQL_PATH),
        ("Video real", REAL_VIDEO_PATH),
        ("Referencia del video", VIDEO_GROUND_TRUTH_PATH),
        ("Imágenes de la garita", E2E_IMAGES_DIR),
        ("Referencia de la garita", E2E_GROUND_TRUTH_CSV),
        ("Dataset propio (YOLO)", LOCAL_DATASET_ZIP),
    )]).set_index("entrada"))
print("La hoja de revisión se lee en la sección 5 con tu cuenta de Google (ID " + REVIEW_SHEET_ID + ").")

Mounted at /content/drive
Carpeta de trabajo: /content/drive/MyDrive/proyecto_integrador_placas


,estado,ruta
entrada,,
SAVH: recortes y ground_truth.csv,ok,/content/drive/MyDrive/proyecto_integrador_pla...
Revisión de placas: copia CSV (opcional),no está,/content/drive/MyDrive/proyecto_integrador_pla...
"SAVH: lecturas de GLM-OCR (comparación, opcional)",no está,/content/drive/MyDrive/proyecto_integrador_pla...
Placas autorizadas (excepciones),vacío,/content/drive/MyDrive/proyecto_integrador_pla...
Base de la planta: PostgreSQL,ok,/content/drive/MyDrive/proyecto_integrador_pla...
Video real,ok,/content/drive/MyDrive/proyecto_integrador_pla...
Referencia del video,no está,/content/drive/MyDrive/proyecto_integrador_pla...
Imágenes de la garita,no está,/content/drive/MyDrive/proyecto_integrador_pla...
Referencia de la garita,no está,/content/drive/MyDrive/proyecto_integrador_pla...


La hoja de revisión se lee en la sección 5 con tu cuenta de Google (ID 1l79GrAYGFkTiOYxgKDT1_BEi_b-6Duo1_QqUNlcv7yM).


### Configuración

Todos los parámetros del experimento están en la celda siguiente, agrupados por tema:

- **Dataset:** origen (`DATASET_SOURCE`) y proyecto y versión de Roboflow.
- **Detector:** pesos iniciales, `batch` y argumentos de inferencia (`infer_kwargs`, vacío: YOLO11 usa el NMS por defecto de Ultralytics). `video_imgsz` es el tamaño al que el detector analiza el video, las imágenes de la garita y la cámara en vivo: 1280 px (la resolución de la cámara); la evaluación en *val* y *test* usa siempre 640, el tamaño de entrenamiento. `CONFIGS` tiene una entrada por configuración del detector, con el nombre que aparece en la base y en las tablas (y en el video, con `SHOW_VIDEO_TITLE = True`); la recorren las secciones 6, 7 y 9.
- **Entrenamiento:** preset, épocas, fracción de los datos, tamaño de imagen, paciencia del *early stopping* y semilla.
- **Umbrales:** piso del umbral calibrado y precisión mínima para decidir sin verificación manual; los umbrales mismos se calibran en la sección 3.
- **Seguimiento y OCR:** cuadros para confirmar un *track*, tamaños mínimos de placa para seguirla y para leerla, modelo de fast-plate-ocr y confianza mínima del OCR.
- **Control de acceso:** formatos válidos de placa.
- **Metas del proyecto:** mAP@0.5 ≥ 0,85, EMR ≥ 80 % y ≤ 1 s por imagen, que aparecen como líneas de referencia en los gráficos.
- **Carpetas de salida y utilidades** comunes: nombre de la corrida, cargar el detector y armar sus argumentos de inferencia (`detector_kwargs`), guardar y leer resultados en JSON.

**Protocolo de entrenamiento.** Dataset y splits fijos, tamaño de imagen 640, semilla 0, *early stopping* con paciencia 10 y lote efectivo de 64 imágenes: Ultralytics acumula gradientes hasta 64 imágenes por paso (`nbs=64`), así que `batch` solo cambia la memoria usada. Con este tamaño de dataset Ultralytics elige AdamW.

**Por qué YOLO11-s.** Es un detector pequeño (≈9,4 M de parámetros y ≈21 GFLOPs a 640 px; mAP@0.5:0.95 de 47,0 en COCO según Ultralytics) que corre rápido incluso en hardware modesto y se exporta a ONNX para correr en CPU dentro de la app. Las placas son una sola clase, mucho más simple que las 80 de COCO, y en *test* supera con holgura la meta de mAP@0.5 (sección 3). Otros tamaños: `yolo11n.pt`, más liviano, o `yolo11m.pt` y `yolo11l.pt`, más precisos en COCO.

**NMS.** YOLO11 propone varias cajas por objeto y se queda con la mejor mediante NMS (supresión de no máximos), con el umbral de solapamiento por defecto de Ultralytics (`iou=0.7`), igual al validar y al predecir. No tiene una cabeza sin NMS como la de YOLO26, el modelo de versiones anteriores de este notebook. Las latencias de la sección 4 incluyen el NMS, en el postproceso.

**Presets.** `rapido` entrena con una muestra aleatoria fija del 30 % de *train* y valida cada época con el 25 % de *val*, en 15 épocas: sirve para probar el flujo completo y tener números preliminares. `completo` usa todo con 40 épocas y es el que va al informe. Mira cuánto tarda la primera época antes de dejarlo corriendo: con las 2 CPU de Colab la carga de datos suele ser el cuello de botella.

**Umbrales.** Los umbrales de confianza no se fijan a mano: se calibran sobre *val* en la sección 3.


In [ ]:
# ===================== Dataset (detalles en la sección 1) =====================
DATASET_SOURCE = "roboflow"  # "roboflow" | "local"
RF_WORKSPACE = "roboflow-universe-projects"
RF_PROJECT = "license-plate-recognition-rxg4e"
RF_VERSION = 11
DATASET_ID = f"{RF_PROJECT}-v{RF_VERSION}" if DATASET_SOURCE == "roboflow" else LOCAL_DATASET_ZIP.stem

# ===================== Detector =====================
DETECTORS = {
    "YOLO11": {
        "loader": YOLO,
        "weights": "yolo11s.pt",         # yolo11n / s / m / l / x
        "batch": 16,
        "deterministic": True,
        "infer_kwargs": {},              # NMS por defecto de Ultralytics (iou=0.7): YOLO11 no tiene cabeza sin NMS
        "video_imgsz": 1280,             # video, garita y en vivo: la resolución de la cámara (a 640 no ve placas chicas)
    },
}
MODEL_COLORS = {"YOLO11": "#2a78d6"}  # color del modelo en todos los gráficos

# ===================== Configuraciones que corren el pipeline (secciones 6, 7 y 9) =====================
# etiqueta → (modelo, cambios en la inferencia, sufijo de los resultados). Una variante de inferencia
# (otros argumentos de predict) se agregaría aquí con su propio sufijo, sin tocar los resultados principales.
CONFIGS = {name: (name, None, "") for name in DETECTORS}

# ===================== Entrenamiento =====================
TRAINING_PRESET = "completo"  # "rapido" para probar el flujo · "completo" para el informe
PRESETS = {
    "rapido":   {"epochs": 15, "train_fraction": 0.30, "val_fraction": 0.25},
    "completo": {"epochs": 40, "train_fraction": 1.00, "val_fraction": 1.00},
}
EPOCHS = PRESETS[TRAINING_PRESET]["epochs"]
TRAIN_FRACTION = PRESETS[TRAINING_PRESET]["train_fraction"]
VAL_FRACTION = PRESETS[TRAINING_PRESET]["val_fraction"]
IMGSZ = 640
PATIENCE = 10          # early stopping: épocas seguidas sin mejorar en val
SEED = 0
WORKERS = 2            # Colab tiene 2 CPU
EVAL_BATCH = 16
FORCE_RETRAIN = False  # True borra el entrenamiento guardado de este preset y lo repite desde cero

# ===================== Umbrales de confianza (se calibran en la sección 3) =====================
CONF_FLOOR = 0.10                 # piso para el umbral calibrado
DECISION_PRECISION_TARGET = 0.95  # precisión mínima en val para aceptar una detección sin verificación manual

# ===================== Seguimiento y OCR (secciones 5 y 6) =====================
MINIMUM_CONSECUTIVE_FRAMES = 10
MIN_PLATE_AREA_RATIO = 0.001  # placas más chicas (fracción del frame) ni se siguen
OCR_PLATE_AREA_RATIO = 0.002  # tamaño mínimo para leer: en video_camiones.mp4 las placas de camiones detenidos miden 0,3–0,5 % del frame
OCR_MODEL_NAME = "cct-s-v2-global-model"  # fast-plate-ocr; "cct-xs-v2-global-model" es más chico y, en SAVH, algo menos preciso
OCR_CONF_MIN = None   # confianza mínima del OCR para decidir sin verificación manual; None = no se usa (sección 5)

# ===================== Control de acceso (sección 5) =====================
# Placas ecuatorianas: 3 letras + 3 o 4 dígitos. Con la revisión de placas, las 472 legibles de SAVH siguen uno
# de esos dos formatos. Los patrones de 2 letras + 4 o 5 dígitos de la v3.5 salían de errores de la cámara en la
# referencia (GB01825 es GBO-1825) y hacían que normalize_plate aceptara esas lecturas sin corregirlas.
PLATE_PATTERNS = [(3, 4), (3, 3)]  # (letras, dígitos), por frecuencia
# Para decidir, guardar y mostrar, las placas válidas se escriben XXX-0000 (format_plate, sección 5).

# ===================== Metas del proyecto (líneas de referencia en los gráficos) =====================
TARGET_MAP50 = 0.85
TARGET_EMR = 0.80
TARGET_E2E_S = 1.0

COLOR = sv.ColorPalette.from_hex([  # colores por track en los videos (de la v2)
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

# ===================== Carpetas de salida (separadas por dataset) =====================
EXPERIMENT_DIR = WORK_DIR / "experimentos" / DATASET_ID
RUNS_DIR = EXPERIMENT_DIR / "runs"           # entrenamientos y validaciones de Ultralytics
RESULTS_DIR = EXPERIMENT_DIR / "resultados"  # métricas por modelo (JSON / CSV) y registros de decisiones
FIGURES_DIR = EXPERIMENT_DIR / "figuras"     # gráficos (PNG) para el informe
VIDEOS_DIR = EXPERIMENT_DIR / "videos"       # videos anotados
OCR_RESULTS_DIR = WORK_DIR / "ocr"           # evaluación del OCR en SAVH (no depende del detector)
for folder in (RUNS_DIR, RESULTS_DIR, FIGURES_DIR, VIDEOS_DIR, OCR_RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Salidas de este experimento:", EXPERIMENT_DIR)


# ===================== Utilidades comunes =====================
def run_name(name: str) -> str:
    """Identificador de la corrida: modelo + pesos + preset, p. ej. 'YOLO11_yolo11s_rapido'."""
    return f"{name}_{Path(DETECTORS[name]['weights']).stem}_{TRAINING_PRESET}"


def best_weights_path(name: str) -> Path:
    return RUNS_DIR / run_name(name) / "weights" / "best.pt"


def trained_models() -> list:
    ready = [name for name in DETECTORS if best_weights_path(name).exists()]
    missing = [name for name in DETECTORS if name not in ready]
    if missing:
        print("Aviso: todavía sin entrenar en este preset:", ", ".join(missing))
    return ready


def load_detector(name: str):
    return DETECTORS[name]["loader"](str(best_weights_path(name)))


def detector_kwargs(name: str, override=None) -> dict:
    """Argumentos de inferencia del modelo (`infer_kwargs` en DETECTORS). `override` los cambia para una variante de
    CONFIGS; un valor None vuelve al valor por defecto de Ultralytics."""
    kwargs = {**DETECTORS[name]["infer_kwargs"], **(override or {})}
    return {key: value for key, value in kwargs.items() if value is not None}


def video_imgsz(name: str) -> int:
    """Tamaño de inferencia en el video, la garita y el modo en vivo (`video_imgsz` en DETECTORS; None = IMGSZ). La
    evaluación en val y test (secciones 3 y 4) usa siempre IMGSZ, el tamaño de entrenamiento."""
    return DETECTORS[name].get("video_imgsz") or IMGSZ


def _to_json(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"No se puede guardar {type(value)} en JSON")


def save_result(name: str, kind: str, data: dict) -> None:
    (RESULTS_DIR / f"{run_name(name)}_{kind}.json").write_text(json.dumps(data, indent=2, default=_to_json))


def load_result(name: str, kind: str):
    path = RESULTS_DIR / f"{run_name(name)}_{kind}.json"
    return json.loads(path.read_text()) if path.exists() else None


def free_gpu() -> None:
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()


def to_detections(result) -> sv.Detections:
    """Resultado de Ultralytics → sv.Detections en float32 (en FP16 las cajas llegan en float16)."""
    detections = sv.Detections.from_ultralytics(result)
    detections.xyxy = detections.xyxy.astype(np.float32)
    if detections.confidence is not None:
        detections.confidence = detections.confidence.astype(np.float32)
    return detections

Salidas de este experimento: /content/drive/MyDrive/proyecto_integrador_placas/experimentos/license-plate-recognition-rxg4e-v11


## 1. Dataset de detección de placas

El detector se entrena y evalúa con un dataset fijo y sus splits. Hay dos fuentes posibles:

- `DATASET_SOURCE = "roboflow"` (por defecto): **License Plate Recognition** de Roboflow Universe (`roboflow-universe-projects/license-plate-recognition-rxg4e`, versión 11: 10 125 imágenes — 7 057 *train*, 2 048 *val*, 1 020 *test* —, una clase, licencia CC BY 4.0). No es específico de Ecuador: sirve para aprender a detectar placas en general. En la celda de configuración puedes cambiar `RF_WORKSPACE`/`RF_PROJECT`/`RF_VERSION` por un proyecto propio.
- `DATASET_SOURCE = "local"`: un `.zip` propio en formato YOLO (`train/`, `valid/`, `test/` con `images/` y `labels/`, más `data.yaml`) en la carpeta de Drive, por ejemplo las imágenes de la garita anotadas en Roboflow y exportadas como *YOLOv8*.

**Para el informe:** lo ideal es que `test` tenga imágenes del dominio real (camiones en la garita). Un mAP alto en un test genérico no garantiza lo mismo con placas ecuatorianas.

La celda reescribe `data.yaml` con rutas absolutas (el de Roboflow trae rutas `../train/images` que no siempre se resuelven bien) y, en el preset `rapido`, crea las listas con la muestra aleatoria fija.

La celda imprime la carpeta del dataset y sus clases.

In [ ]:
# Disco local de Colab: leer miles de imágenes desde Drive en cada época sería mucho más lento.
DATASET_DIR = Path("/content/datasets") / DATASET_ID

# Obtener la clave de API de Roboflow desde los Secretos de Colab
ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY")

def get_dataset() -> Path:
    """Descarga o descomprime el dataset (una sola vez por sesión) y devuelve la carpeta con data.yaml."""
    if not list(DATASET_DIR.rglob("data.yaml")):
        if DATASET_SOURCE == "roboflow":
            if not ROBOFLOW_API_KEY:
                raise RuntimeError("Falta el secreto ROBOFLOW_API_KEY (sección 0).")
            from roboflow import Roboflow
            DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
            project = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(RF_WORKSPACE).project(RF_PROJECT)
            project.version(RF_VERSION).download("yolov8", location=str(DATASET_DIR), overwrite=True)
        elif DATASET_SOURCE == "local":
            with zipfile.ZipFile(LOCAL_DATASET_ZIP) as archive:
                archive.extractall(DATASET_DIR)
        else:
            raise ValueError("DATASET_SOURCE debe ser 'roboflow' o 'local'")
    return min(DATASET_DIR.rglob("data.yaml"), key=lambda p: len(p.parts)).parent


def write_data_yaml(root: Path) -> Path:
    """data.yaml con rutas absolutas y los splits que existan (Roboflow llama 'valid' al de validación)."""
    original = yaml.safe_load((root / "data.yaml").read_text())
    names = original.get("names", ["placa"])
    if isinstance(names, list):
        names = dict(enumerate(names))
    data = {"path": str(root.resolve()), "names": names}
    for key, candidates in {"train": ["train"], "val": ["valid", "val"], "test": ["test"]}.items():
        folder = next((c for c in candidates if (root / c / "images").is_dir()), None)
        if folder:
            data[key] = f"{folder}/images"
    if "test" not in data:
        print("Aviso: el dataset no trae split 'test'; se evaluará sobre val (no es lo ideal).")
        data["test"] = data["val"]
    if len(names) > 1:
        print(f"Aviso: el dataset tiene {len(names)} clases; el pipeline asume que todas son placas.")
    out = root / "data_abs.yaml"
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return out


def list_images(folder: Path) -> list:
    return sorted(p for p in Path(folder).iterdir() if p.suffix[1:].lower() in IMG_FORMATS)


def write_subset_yaml(data_yaml: Path) -> Path:
    """Preset 'rapido': muestra aleatoria fija de train y val, la misma en cada corrida."""
    if TRAIN_FRACTION >= 1 and VAL_FRACTION >= 1:
        return data_yaml
    data = yaml.safe_load(data_yaml.read_text())
    root, rng = Path(data["path"]), random.Random(SEED)
    for key, fraction in (("train", TRAIN_FRACTION), ("val", VAL_FRACTION)):
        images = list_images(root / data[key])
        chosen = sorted(rng.sample(images, max(1, round(len(images) * fraction))))
        listing = root / f"{key}_{TRAINING_PRESET}.txt"
        listing.write_text("\n".join(str(p) for p in chosen) + "\n")
        data[key] = str(listing)
    out = root / f"data_{TRAINING_PRESET}.yaml"
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    return out


DATASET_ROOT = get_dataset()
DATA_YAML = write_data_yaml(DATASET_ROOT)       # evaluación: val y test completos
TRAIN_DATA_YAML = write_subset_yaml(DATA_YAML)  # entrenamiento: muestra fija en el preset 'rapido'
DATA = yaml.safe_load(DATA_YAML.read_text())
print("Dataset:", DATASET_ROOT, "| clases:", DATA["names"])

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/datasets/license-plate-recognition-rxg4e-v11 in yolov8:: 100%|██████████| 20262/20262 [00:02<00:00, 9061.05it/s] 


Dataset: /content/datasets/license-plate-recognition-rxg4e-v11 | clases: {0: 'License_Plate'}


### Normalización de placas
El texto del OCR no siempre es una placa válida. Antes de decidir el acceso hace falta encontrar la placa en el texto, corregir confusiones típicas por posición y validar el formato. fast-plate-ocr ya devuelve solo letras y dígitos, casi siempre con forma de placa: en SAVH, 398 de sus 399 lecturas de legibles ya tenían un formato válido y la normalización corrigió una sola (`GP1226` → `GPI226`). Pero la misma normalización se aplica también a la lectura de la cámara en SAVH y al texto libre de GLM-OCR en la comparación de más abajo; con GLM-OCR subía el EMR de 78 % a 93 %.

Las placas ecuatorianas tienen **3 letras + 3 o 4 dígitos** ([El Universo](https://www.eluniverso.com/noticias/2020/06/26/nota/7885368/placas-carros-como-identificarlas/)). La v3.5 aceptaba además 2 letras + 4 o 5 dígitos, pero esos formatos salían de errores de la cámara en la referencia de SAVH (`GB01825` es `GBO-1825`), así que solo se usan los dos formatos reales.

`normalize_plate` hace tres pasos:

1. **Busca la placa dentro del texto.** Corta el texto en tramos (por espacios, guiones, saltos de línea y otra puntuación) y prueba cada tramo y las uniones de 2 o 3 tramos seguidos (`GBA 1234`). Descarta la leyenda `ECUADOR` impresa sobre la placa: en la corrida de la v3.6, GLM-OCR la transcribió en 16 de las 399 legibles de SAVH, a veces mal leída (`EQUADOR`, `ECUNDERG`), y como la limpieza la pegaba a la placa las 16 contaban como error. fast-plate-ocr no escribe separadores ni la leyenda, así que con él este paso no cambia nada.
2. **Corrige por posición.** En las 3 primeras posiciones (letras) cambia un dígito por la letra parecida (0→O, 1→I, 5→S, 6→G, 8→B) y en las siguientes (dígitos) una letra por el dígito parecido (O→0, D→0, Q→0, I→1, Z→2, B→8); `|` se lee como 1 y `$` como S. Gana el candidato que valida con menos cambios, y se aceptan como máximo 2: en SAVH, ninguna lectura que necesitó 3 era correcta.
3. **Valida el formato.** Si ningún candidato valida, la lectura queda inválida y la regla de decisión la manda a verificación manual.

La v3.6 corregía además G→6, pero GLM-OCR escribía `G` en lugar de 0 (`GTT-187G` es `GTT-1870`) y también de 6. Es mejor que esas lecturas vayan a verificación manual que convertirlas en otra placa válida. Por la misma razón, un símbolo que no es separador (`=`, `Φ`, `&`) invalida su tramo. La celda termina con una prueba sobre lecturas reales de la cámara y de GLM-OCR. Las motos tienen otro formato y no aparecen en SAVH; si hicieran falta, habría que agregar su patrón.

**Formato para decidir, guardar y mostrar.** `format_plate` escribe cada placa válida como en Ecuador: 3 letras, guion y 4 dígitos (`GBA-1234`). A las placas antiguas de 3 dígitos les agrega un 0 al inicio de los números (`GMS-429` → `GMS-0429`), para que todas queden en el formato XXX-0000; por eso `GMS-429` y `GMS-0429` cuentan como la misma placa. Ese es el formato de la lista de autorizados, de la regla de decisión, de la base de datos de la planta, de los registros y del video. La evaluación del OCR con SAVH compara las placas sin guion, porque mide la lectura y no el formato.

In [ ]:
CONFUSIONS_TO_LETTER = {"0": "O", "1": "I", "5": "S", "6": "G", "8": "B"}  # dígito leído donde va una letra
CONFUSIONS_TO_DIGIT = {"O": "0", "D": "0", "Q": "0", "I": "1", "Z": "2", "B": "8"}  # letra leída donde va un dígito
SYMBOLS_TO_CHAR = {"|": "1", "$": "S"}  # símbolos que GLM-OCR escribía en lugar de un carácter
SEPARATORS = " \t\r\n-‐‑‒–—_.·•:,;+\"'`*()[]{}<>"  # cortan el texto; cualquier otro símbolo invalida su tramo
MAX_CORRECTIONS = 2       # en SAVH, ninguna lectura que necesitó 3 cambios era correcta
PLATE_HEADER = "ECUADOR"  # leyenda impresa sobre la placa; GLM-OCR a veces la transcribía en la primera línea
_SEGMENT = re.compile("[^" + re.escape(SEPARATORS) + "]+")


def clean_plate_text(text) -> str:
    """Mayúsculas y solo letras y dígitos: 'gba-1234' → 'GBA1234'."""
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())


def is_plate_header(segment: str) -> bool:
    """'ECUADOR' y sus lecturas aproximadas ('EQUADOR', 'ECUNDERG'): solo letras y a 3 cambios o menos."""
    return (len(segment) >= 5 and segment.isascii() and segment.isalpha()
            and Levenshtein.distance(segment, PLATE_HEADER) <= 3)


def as_plate(candidate: str, corrections: bool = True):
    """(placa, cambios) si el candidato tiene formato de placa, corrigiendo por posición si hace falta; si no, None."""
    for n_letters, n_digits in PLATE_PATTERNS:
        if len(candidate) != n_letters + n_digits:
            continue
        letters, digits = candidate[:n_letters], candidate[n_letters:]
        if corrections:
            letters = "".join(CONFUSIONS_TO_LETTER.get(c, c) for c in letters)
            digits = "".join(CONFUSIONS_TO_DIGIT.get(c, c) for c in digits)
        plate = letters + digits
        changes = sum(a != b for a, b in zip(plate, candidate))
        if changes <= MAX_CORRECTIONS and re.fullmatch(f"[A-Z]{{{n_letters}}}[0-9]{{{n_digits}}}", plate):
            return plate, changes
    return None


def find_plate(raw_text, search: bool = True, corrections: bool = True) -> tuple:
    """Busca la placa en el texto del OCR → (placa, formato válido, cambios, (inicio, fin) en el texto).

    Prueba cada tramo entre separadores y las uniones de 2 o 3 tramos seguidos ('GBA 1234'), sin la leyenda
    ECUADOR. Gana el candidato que valida con menos cambios y, a igualdad, el que aparece primero. Si ninguno
    valida, devuelve el texto limpio sin la leyenda, marcado como inválido. `search=False` y `corrections=False`
    apagan cada paso, para la ablación."""
    text = str(raw_text).upper()
    if not search:  # el texto entero como un solo candidato
        found = as_plate(clean_plate_text(text), corrections)
        return (found[0], True, found[1], None) if found else (clean_plate_text(text), False, 0, None)
    segments = [(m.start(), m.end(), "".join(SYMBOLS_TO_CHAR.get(c, c) for c in m.group()))
                for m in _SEGMENT.finditer(text)]
    segments = [s for s in segments if not is_plate_header(s[2])]
    best = None
    for i in range(len(segments)):
        for j in range(i + 1, min(i + 3, len(segments)) + 1):
            found = as_plate("".join(s[2] for s in segments[i:j]), corrections)
            if found and (best is None or found[1] < best[2]):
                best = (found[0], True, found[1], (segments[i][0], segments[j - 1][1]))
    return best or (clean_plate_text("".join(s[2] for s in segments)), False, 0, None)


def normalize_plate(raw_text: str) -> tuple[str, bool]:
    """Texto del OCR → (placa normalizada, formato válido)."""
    plate, is_valid, _, _ = find_plate(raw_text)
    return plate, is_valid


def format_plate(text) -> str:
    """Formato de Ecuador para decidir, guardar y mostrar: 3 letras, guion y 4 dígitos ('gba1234' → 'GBA-1234'). A las
    placas antiguas de 3 dígitos se les agrega un 0 al inicio de los números ('GMS-429' → 'GMS-0429'). Un texto que no
    es una placa válida queda limpio (mayúsculas, solo letras y dígitos), sin guion."""
    plate = clean_plate_text(text)
    match = re.fullmatch(r"([A-Z]{3})(\d{3,4})", plate)
    return f"{match.group(1)}-{match.group(2).zfill(4)}" if match else plate


def plate_confidence(text: str, char_probs) -> float:
    """Confianza del OCR en la placa: la menor probabilidad entre sus letras y dígitos dentro del texto
    (sin placa válida, entre todas las letras y dígitos). NaN si no hay probabilidades."""
    if char_probs is None or len(char_probs) != len(text):
        return float("nan")
    span = find_plate(text)[3]
    start, end = span if span and len(text.upper()) == len(text) else (0, len(text))
    probs = [p for c, p in zip(text[start:end], char_probs[start:end]) if c.isalnum() or c in SYMBOLS_TO_CHAR]
    return float(min(probs)) if probs else 0.0


# Lecturas reales: de la cámara en SAVH (revisión de placas) y de GLM-OCR en la corrida de la v3.6.
for raw, expected in [("GB01825", "GBO1825"), ("GS11034", "GSI1034"), ("6TK6566", "GTK6566"), ("GTE5D45", "GTE5045"),
                      ("GP1986", "GPI986"), ("gba-1234", "GBA1234"), ("ECUADOR\nGMS-429", "GMS429"),
                      ("EQUIDOR\nGTE-7040", "GTE7040"), ("ECUNDERG\nGB0-6445", "GBO6445"), ("G5V-3233", "GSV3233"),
                      ("G50-6164", "GSO6164"), ("G|G-430", "GIG430"), ("GSH+2315", "GSH2315"), ("GTT·4568", "GTT4568"),
                      ("ECUADOR GMS 429", "GMS429")]:
    normalized, is_valid = normalize_plate(raw)
    assert is_valid and normalized == expected, f"{raw!r} → {normalized} (se esperaba {expected})"
# Lecturas que deben quedar inválidas (→ verificación manual) en vez de convertirse en otra placa válida.
for raw in ["GTT-187G", "GPA-1G88", "OBX=660", "GBΦ6184", "115220", "ECUADOR", ""]:
    normalized, is_valid = normalize_plate(raw)
    assert not is_valid, f"{raw!r} no debería validar (salió {normalized})"
assert plate_confidence("ECUADOR\nGMS-429", [0.2] * 8 + [0.9, 0.8, 0.95, 0.3, 0.99, 0.7, 0.9]) == 0.7
for raw, expected in [("gba1234", "GBA-1234"), ("GBA-1234", "GBA-1234"), ("GMS429", "GMS-0429"), ("gms-429", "GMS-0429"),
                      ("PBA 123", "PBA-0123"), ("PBA-0123", "PBA-0123"), ("G8A1234", "G8A1234"), ("", "")]:
    assert format_plate(raw) == expected, f"{raw!r} → {format_plate(raw)!r} (se esperaba {expected!r})"
print("normalize_plate y format_plate: los casos de prueba pasan.")

normalize_plate y format_plate: los casos de prueba pasan.
